# ML Column Mapping Example (Semantic Embeddings)

This notebook demonstrates how to map messy source columns (e.g., from client Excel uploads) to your standard target columns in your Legal Mapping system using Natural Language Processing (`sentence-transformers`).

In [147]:
# Run this cell to install dependencies (if you haven't already)
%pip install sentence-transformers pandas thefuzz python-Levenshtein

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [148]:
import warnings
warnings.filterwarnings('ignore')

from sentence_transformers import SentenceTransformer, util
from thefuzz import fuzz, process
import pandas as pd

# 1. Load the pre-trained ML model (this will take a moment to download the first time)
print("Loading Sentence Transformer model...")
model = SentenceTransformer('all-MiniLM-L6-v2')
print("Model loaded!")

Loading Sentence Transformer model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 251.68it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded!


### Step 2: Define Targets & Abbreviation Dictionary
ML models don't naturally understand abbreviations (e.g. they don't know 'RM' = 'Regional Manager'). We create a dictionary to "clean" the data before asking the AI.

In [ ]:
target_columns = [
    "full_name",
    "dpd",
    "total_outstanding_amt",
    "email",
    "phone_num",
    "address",
    "lan",
    "office_Address",
    "pincode",
    "language",
    "state",
    "loan_amount",
    "regional_manager",
    "regional_manager_phone_number",
    "phone_number",
    "mobile_number",
    "agreement_date",
    "city",
    "notice",
    "outstanding_amount",
    "store",
    "collection_manager",
    "collection_manager_phone_number"
]

# Dictionary to map standard industry abbreviations to their full English words
abbreviation_dict = {
    'rm': 'regional manager',
    'acm': 'collection manager',
    'lan': 'loan account number',
    'dpd': 'days past due',
    'amt': 'amount',
    'num': 'number',
    'no': 'number',
    'zip': 'pincode', # Sometimes clients say zip code instead of pincode
    'mobile':'phone number',
    'contact':'phone number',
    'location':'address'
}

def clean_and_expand_column_name(col_name):
    """Converts abbreviations to full words (e.g., 'rm_name' -> 'regional manager name')"""
    # Replace underscores and slashes with spaces, then split into words
    words = str(col_name).lower().replace('_', ' ').replace('/', ' ').split()
    # Swap abbreviations if they exist in our dictionary
    expanded_words = [abbreviation_dict.get(word, word) for word in words]
    return ' '.join(expanded_words)

# 3. Create embeddings for your target columns (Note: We embed the CLEANED versions!)
clean_targets = [clean_and_expand_column_name(col) for col in target_columns]
target_embeddings = model.encode(clean_targets)
print("\u2705 Target columns cleaned and mapped to vector embeddings!")

✅ Target columns cleaned and mapped to vector embeddings!


### Approach 1: Fuzzy Matching (Baseline)
Good for finding slight misspellings, but fails on synonyms.

In [150]:
def fuzzy_match_column(source_col):
    # Uses string similarity (Levenshtein distance)
    best_match = process.extractOne(source_col, target_columns, scorer=fuzz.token_sort_ratio)
    return best_match[0], best_match[1] / 100.0  # Name and Confidence (0 to 1)

### Approach 2: Semantic Embeddings (ML)
Understands the *meaning* of words, even if they are spelled completely differently.

In [151]:
def semantic_match_column(raw_source_column_name):
    """Predicts the best target column using deep learning embeddings."""
    # Clean the incoming messy column before asking the AI
    clean_source_col = clean_and_expand_column_name(raw_source_column_name)
    
    source_embedding = model.encode(clean_source_col)
    cosine_scores = util.cos_sim(source_embedding, target_embeddings)[0]
    
    best_match_index = cosine_scores.argmax().item()
    confidence_score = cosine_scores[best_match_index].item()
    best_match_name = target_columns[best_match_index] # Return the original target name
    
    return best_match_name, confidence_score

In [152]:
# Let's test it with some messy source column names you might get in an Excel file
messy_source_columns = [
    "customer_name",           # Synonym for full_name
    "days_past_due",           # Automatically swapped to 'dpd' by dict
    "total_debt",              # Synonym for total_outstanding_amt
    "email_address",           # Synonym for email
    "contact_number",          # Synonym for phone_num/phone_number/mobile_number
    "residential_location",    # Synonym for address
    "loan_account_number",     # Automatically swapped to 'lan' by dict
    "work_address",            # Synonym for office_Address
    "zip_code",                # Automatically mapped to pincode by dict
    "spoken_language",         # Synonym for language
    "region_state",            # Synonym for state
    "principal_amount",        # Synonym for loan_amount
    "rm_name",                 # Automatically swapped to 'regional manager name'
    "rm_contact",              # Automatically swapped to 'regional manager contact'
    "date_of_agreement",       # Synonym for agreement_date
    "town",                    # Synonym for city
    "legal_notice",            # Synonym for notice
    "balance_amount",          # Synonym for outstanding_amount
    "branch",                  # Synonym for store
    "acm_name",                # Automatically swapped to 'collection manager name'
    "acm_mobile"               # Automatically swapped to 'collection manager mobile'
]

results = []
for col in messy_source_columns:
    fuzzy_col, fuzzy_conf = fuzzy_match_column(col)
    sem_col, sem_conf = semantic_match_column(col)
    
    # We can use a hybrid approach (if fuzzy match is very high, use it, else use semantic)
    final_prediction = fuzzy_col if fuzzy_conf > 0.85 else sem_col
    final_confidence = fuzzy_conf if fuzzy_conf > 0.85 else sem_conf
    
    results.append({
        "Source (Messy)": col,
        "Winner (Prediction)": final_prediction,
        "Semantic Truth Score": f"{round(sem_conf, 2)}"
    })

# Display the results nicely in a DataFrame
pd.set_option('display.max_rows', None)
df_results = pd.DataFrame(results)
df_results

,Source (Messy),Winner (Prediction),Semantic Truth Score
0,customer_name,full_name,0.45
1,days_past_due,dpd,1.0
2,total_debt,total_outstanding_amt,0.64
3,email_address,email,0.86
4,contact_number,mobile_number,1.0
5,residential_location,address,0.7
6,loan_account_number,lan,1.0
7,work_address,office_Address,0.77
8,zip_code,pincode,0.95
9,spoken_language,language,0.8


## Section: How an ML Model Learns from its Mistakes (Online Learning)

In this section, we'll simulate a scenario where our model makes an incorrect prediction, and we immediately correct it using `partial_fit` (Online Learning).

We will use an `SGDClassifier` which is perfect for this! It allows us to update the model on-the-fly when it makes a mistake, without retraining on the whole dataset.

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier

# 1. Let's create our initial training data
initial_data = pd.DataFrame({
    'column_name': ['Client Name', 'DOB', 'Case Status', 'Payment Amount'],
    'standard_field': ['Plaintiff', 'Date of Birth', 'Status', 'Amount']
})

vectorizer = TfidfVectorizer()
X_train = vectorizer.fit_transform(initial_data['column_name'])
y_train = initial_data['standard_field']

# Initialize the classifier (SGDClassifier supports online learning with partial_fit)
model = SGDClassifier(loss='log_loss', random_state=42)

# We need to provide all possible classes it might ever see to partial_fit initially
all_possible_classes = ['Plaintiff', 'Date of Birth', 'Status', 'Amount', 'Defendant Name']
model.partial_fit(X_train, y_train, classes=all_possible_classes)

print("✅ Model initially trained!")

# 2. New column comes in that the model hasn't seen
new_column = "Def. Name"
X_new = vectorizer.transform([new_column])

# The model makes a guess (Prediction)
prediction = model.predict(X_new)[0]
print(f"\n🤔 Model guesses '{new_column}' maps to: '{prediction}'")
print("❌ Uh oh! That's a mistake. It should be 'Defendant Name'.")

# 3. Learning from the mistake!
correct_label = "Defendant Name"
print(f"\n👨‍🏫 User Correction: Teaching the model that '{new_column}' is actually '{correct_label}'...")

# Update the model with this single new example using partial_fit
model.partial_fit(X_new, [correct_label])

# 4. Let's check if it learned!
new_prediction = model.predict(X_new)[0]
print(f"\n🧠 Model's NEW guess for '{new_column}' matches: '{new_prediction}'")
if new_prediction == correct_label:
    print("🎯 The model successfully learned from its mistake!")


## Section: Incorporating Frontend Rules into ML

This section maps the exact rules defined in your frontend's `headerMatcher.ts` into our ML model. By combining explicit rules (like `KNOWN_MAPPINGS`) and standardizing text (like `normalizeHeader`), we give our ML model the same base knowledge your application currently uses.

In [ ]:
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

# 1. Define Frontend Rules (from headerMatcher.ts)
TARGET_HEADERS = [
    'sno', 'Account Number', 'LAN', 'TOTdate', 'disp_amt', 'TOT', 'full name', 
    'co_borrower', 'Pincode', 'Product', 'Product type', 'language', 'Language 1', 
    'Language 2', 'barcode', 'state', 'loan amount', 'ACM phone number', 
    'phone number', 'Mobile Number', 'ACM name', 'Agreement date', 'email', 
    'address', 'city', 'DPD', 'Notice', 'Outstanding Amount', 'Total Outstanding Amount'
]

KNOWN_MAPPINGS = {
    'lone account number': 'LAN',
    'lone acct no': 'LAN',
    'loan acct no': 'LAN',
    'lan': 'LAN',
    'acc no': 'Account Number',
    'acct no': 'Account Number',
    'mob': 'Mobile Number',
    'ph': 'phone number',
    'addr': 'address',
}

# 2. Define text normalization matching frontend
def normalize_header(header):
    if not isinstance(header, str):
        return ""
    h = header.lower()
    h = re.sub(r'[^a-z0-9]', ' ', h) # Replace specials with space
    h = re.sub(r'\s+', ' ', h).strip() # Collapse spaces
    h = h.replace('lone', 'loan')    # Fix specific typo
    return h

def apply_normalization(X):
    return [normalize_header(x) for x in X]

# 3. Create Training Data combining Target Headers and Known Mappings
train_texts = TARGET_HEADERS + list(KNOWN_MAPPINGS.keys())
train_labels = TARGET_HEADERS + list(KNOWN_MAPPINGS.values())

# 4. Build the ML Pipeline
pipeline = Pipeline([
    ('normalizer', FunctionTransformer(apply_normalization)),
    ('vectorizer', TfidfVectorizer(ngram_range=(1, 2))),  # Use unigrams & bigrams
    ('classifier', SGDClassifier(loss='log_loss', random_state=42))
])

# 5. Train the Model
pipeline.fit(train_texts, train_labels)
print("✅ ML Model trained with all frontend rules and targets!")

# 6. Test with known frontend edge cases
test_cases = ['lone acct no ', 'Acc. No.', 'mob', 'Client addr', 'S.No.']
predictions = pipeline.predict(test_cases)
probabilities = pipeline.predict_proba(test_cases)

print("\n--- Testing Frontend Edge Cases ---")
for text, pred, prob in zip(test_cases, predictions, probabilities):
    max_prob = max(prob)
    print(f"{text: <15} -> {pred: <18} (Confidence: {max_prob:.2f})")
